In [1]:
import os
import sys
import json
import hashlib
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# TODO: Add these imports
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from huggingface_hub import snapshot_download, HfApi
import importlib.metadata

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

print("Working directory:", os.getcwd())
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", importlib.metadata.version("transformers"))

Working directory: /home/alireza/PycharmProjects/Quera-Homeworks/HW3_Student/HW3_A
Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
Torch: 2.12.1+cu130
Transformers: 5.12.1


# Step 1: Download model from HuggingFace

Download `sentence-transformers/all-MiniLM-L6-v2` and save 6 files to `bundle/model/`:

| # | File |
|---|------|
| 1 | `config.json` |
| 2 | `tokenizer_config.json` |
| 3 | `tokenizer.json` |
| 4 | `vocab.txt` |
| 5 | `special_tokens_map.json` |
| 6 | `model.safetensors` |

Use `snapshot_download` from `huggingface_hub` or download manually with `AutoModel.save_pretrained()` + `AutoTokenizer.save_pretrained()`.

After downloading, save the git commit hash to `bundle/model/.commit`.

In [20]:
# TODO: Download the 6 model files to bundle/model/

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
BUNDLE_MODEL_DIR = Path("bundle/model")
BUNDLE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# HINT: Use snapshot_download(repo_id=MODEL_ID, revision="main",
#        local_dir=str(BUNDLE_MODEL_DIR), allow_patterns=[...])
# HINT: Required files: config.json, tokenizer_config.json, tokenizer.json,
#        vocab.txt, special_tokens_map.json, model.safetensors

# --- YOUR CODE HERE ---
snapshot_download(
    repo_id=MODEL_ID,
    revision="main",
    local_dir=str(BUNDLE_MODEL_DIR),
    allow_patterns=[
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "vocab.txt",
        "special_tokens_map.json",
        "model.safetensors",
    ],
)
# --- END YOUR CODE ---

# Save commit hash
# HINT: from huggingface_hub import HfApi
# HINT: commit = HfApi().model_info(MODEL_ID, revision="main").sha
# --- YOUR CODE HERE ---
commit = HfApi().model_info(MODEL_ID, revision="main").sha
(BUNDLE_MODEL_DIR / ".commit").write_text(commit + "\n", encoding="utf-8")

print("Model ID:", MODEL_ID)
print("Commit:", commit)
print("Saved to:", BUNDLE_MODEL_DIR.resolve())
# --- END YOUR CODE ---

# Verify all 6 files exist
REQUIRED = ["config.json", "tokenizer_config.json", "tokenizer.json",
            "vocab.txt", "special_tokens_map.json", "model.safetensors"]
for fname in REQUIRED:
    fpath = BUNDLE_MODEL_DIR / fname
    assert fpath.exists(), f"MISSING: {fname}"
    size_mb = fpath.stat().st_size / (1024 * 1024)
    print(f"  [OK] {fname:35s} {size_mb:7.1f} MB")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model ID: sentence-transformers/all-MiniLM-L6-v2
Commit: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Saved to: /home/alireza/PycharmProjects/Quera-Homeworks/HW3_Student/HW3_A/bundle/model
  [OK] config.json                             0.0 MB
  [OK] tokenizer_config.json                   0.0 MB
  [OK] tokenizer.json                          0.4 MB
  [OK] vocab.txt                               0.2 MB
  [OK] special_tokens_map.json                 0.0 MB
  [OK] model.safetensors                      86.7 MB


# Step 2: Write metadata.json

Create `bundle/metadata.json` with accurate information about the bundle.
Required fields:
- `model_name`: the HuggingFace model ID
- `model_revision`: the git commit hash from the `.commit` file
- `embedding_dim`: 384
- `max_seq_len`: 256
- `framework_version`: the installed torch version
- `transformers_version`: the installed transformers version
- `built_by`: YOUR NAME
- `build_timestamp_utc`: current time in ISO 8601 UTC format

In [21]:
# TODO: Create bundle/metadata.json

# HINT: commit = (BUNDLE_MODEL_DIR / ".commit").read_text().strip()
# HINT: ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
# HINT: torch_version = torch.__version__
# HINT: transformers_version = importlib.metadata.version("transformers")

# --- YOUR CODE HERE ---
import importlib.metadata

commit = (BUNDLE_MODEL_DIR / ".commit").read_text(encoding="utf-8").strip()
ts = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

torch_version = torch.__version__
transformers_version = importlib.metadata.version("transformers")

metadata = {
    "model_name": MODEL_ID,
    "model_revision": commit,
    "embedding_dim": 384,
    "max_seq_len": 256,
    "framework_version": torch_version,
    "transformers_version": transformers_version,
    "built_by": "Alireza Abavi",
    "build_timestamp_utc": ts,
}
# --- END YOUR CODE ---

# Write to file
meta_path = Path("bundle/metadata.json")
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(json.dumps(metadata, indent=2))

{
  "model_name": "sentence-transformers/all-MiniLM-L6-v2",
  "model_revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
  "embedding_dim": 384,
  "max_seq_len": 256,
  "framework_version": "2.12.1+cu130",
  "transformers_version": "5.12.1",
  "built_by": "Alireza Abavi",
  "build_timestamp_utc": "2026-06-20T17:59:51Z"
}


# Step 3: Write MANIFEST.json

Compute SHA-256 hash for every file under `bundle/` (except MANIFEST.json itself)
and write the manifest to `bundle/MANIFEST.json`.

The format must be:
```json
{
  "format_version": 1,
  "files": {
    "relative/path/to/file": "sha256hexdigest",
    ...
  }
}
```

Pro tip: you can peek at `scripts/gen_manifest.py` for reference.

In [22]:
# TODO: Hash every file in bundle/ and create MANIFEST.json

# HINT: def sha256(filepath):
# HINT:     h = hashlib.sha256()
# HINT:     with open(filepath, "rb") as f:
# HINT:         for chunk in iter(lambda: f.read(1 << 20), b""):
# HINT:             h.update(chunk)
# HINT:     return h.hexdigest()

# --- YOUR CODE HERE ---

def sha256(filepath: Path) -> str:
    h = hashlib.sha256()

    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)

    return h.hexdigest()

# --- END YOUR CODE ---

# Build and write manifest
# HINT: root = Path("bundle")
# HINT: for p in sorted(root.rglob("*")):
# HINT:     if p.is_file() and p.name != "MANIFEST.json":
# HINT:         rel = str(p.relative_to(root))
# HINT:         files[rel] = sha256(p)
# --- YOUR CODE HERE ---

root = Path("bundle")
files = {}

for p in sorted(root.rglob("*")):
    if p.is_file() and p.name != "MANIFEST.json":
        rel = str(p.relative_to(root))
        files[rel] = sha256(p)

manifest = {
    "files": files
}

# --- END YOUR CODE ---

# Write
manifest_path = Path("bundle/MANIFEST.json")
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"MANIFEST.json written with {len(manifest['files'])} files")
for rel, h in sorted(manifest["files"].items()):
    print(f"  {h[:16]}...  {rel}")

MANIFEST.json written with 20 files
  90bde439ae835cbe...  __pycache__/predict.cpython-312.pyc
  c0f2f97c6440584c...  metadata.json
  684888c0ebb17f37...  model/.cache/huggingface/.gitignore
  f6572428f6d5e157...  model/.cache/huggingface/CACHEDIR.TAG
  c6f1643dbe6c6d86...  model/.cache/huggingface/download/config.json.metadata
  d08d11f0149e7135...  model/.cache/huggingface/download/model.safetensors.metadata
  51df7a8df54eb388...  model/.cache/huggingface/download/special_tokens_map.json.metadata
  2c09ca167169503a...  model/.cache/huggingface/download/tokenizer.json.metadata
  083bc8dafcd9e6d6...  model/.cache/huggingface/download/tokenizer_config.json.metadata
  ffed5e669a695b93...  model/.cache/huggingface/download/vocab.txt.metadata
  9d73db3316b1ee7a...  model/.commit
  e3b0c44298fc1c14...  model/.gitkeep
  953f9c0d463486b1...  model/config.json
  53aa51172d142c89...  model/model.safetensors
  303df45a03609e4e...  model/special_tokens_map.json
  be50c3628f2bf5bb...  model/tokeni

# Step 4: Write predict.py

Write `bundle/predict.py` with exactly 4 functions:

| Function | Signature | Returns |
|----------|-----------|---------|
| `load_bundle()` | no args | `(model, tokenizer)` tuple |
| `embed(texts)` | `List[str]` | `np.ndarray` shape `(N, 384)` float32 |
| `similarity(a, b)` | two `np.ndarray` | `float` (cosine similarity) |
| `info()` | no args | `dict` with metadata |

The **7-step pipeline** inside `embed()`:
1. Tokenize: `tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")`
2. Move tensors to device (cpu or cuda)
3. Forward pass under `torch.no_grad()` → `last_hidden_state`
4. Mean-pool: `sum(H * mask) / sum(mask).clamp(min=1e-9)`
5. L2 normalize: `F.normalize(pooled, p=2, dim=1)`
6. Detach, move to CPU, convert to `np.float32`
7. Return ndarray

**Important rules:**
- DO NOT import `sentence-transformers`. Use raw `transformers` only.
- Set `torch.manual_seed(0)` for determinism.
- Call `model.eval()` before inference.
- A template already exists at `bundle/predict.py` — open it and fill in the TODOs.

In [5]:
# TODO: Write the implementation to bundle/predict.py
#
# This cell writes bundle/predict.py with exactly 4 public functions:
#   load_bundle(), embed(), similarity(), info()
#
# Then it imports the module and verifies it works.

predict_path = Path("bundle/predict.py")

predict_path.write_text(r'''#!/usr/bin/env python
"""predict.py — Self-contained embedding inference.

MUST implement exactly 4 functions:
    load_bundle()    -> (model, tokenizer)
    embed(texts)     -> np.ndarray shape (N, 384)
    similarity(a, b) -> float
    info()           -> dict

DO NOT import sentence_transformers. Use raw transformers only.
"""
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEFAULT_MODEL_DIR = Path(__file__).resolve().parent / "model"

# Public constant expected by tests.
BUNDLE_DIR = str(DEFAULT_MODEL_DIR)

MAX_SEQ_LEN = 256

# Public constant expected by tests.
EMBEDDING_DIM = 384

_MODEL = None
_TOKENIZER = None
_DEVICE = None
_LOADED_FROM = None


def _resolve_model_dir(bundle_dir: str | None = None) -> Path:
    """Resolve the model directory safely.

    Important:
    - If bundle_dir is passed directly, use it.
    - Else, if env var BUNDLE_DIR exists, use it.
    - Else, use the default bundle/model directory.

    We do NOT permanently overwrite the public BUNDLE_DIR constant from env,
    because tests may temporarily set BUNDLE_DIR to an invalid path.
    """
    if bundle_dir is not None:
        return Path(bundle_dir).expanduser().resolve()

    env_bundle_dir = os.getenv("BUNDLE_DIR")
    if env_bundle_dir:
        return Path(env_bundle_dir).expanduser().resolve()

    return Path(BUNDLE_DIR).expanduser().resolve()


def load_bundle(bundle_dir: str | None = None) -> Tuple:
    """Load model and tokenizer from the frozen bundle model directory."""
    global _MODEL, _TOKENIZER, _DEVICE, _LOADED_FROM

    model_dir = _resolve_model_dir(bundle_dir)

    required_files = [
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "vocab.txt",
        "special_tokens_map.json",
        "model.safetensors",
    ]

    missing = [fname for fname in required_files if not (model_dir / fname).exists()]
    if missing:
        raise FileNotFoundError(
            f"Invalid bundle model directory: {model_dir}. "
            f"Missing files: {missing}"
        )

    if (
        _MODEL is not None
        and _TOKENIZER is not None
        and _LOADED_FROM == str(model_dir)
    ):
        return _MODEL, _TOKENIZER

    torch.manual_seed(0)
    torch.set_num_threads(int(os.getenv("TORCH_NUM_THREADS", "2")))

    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(
        str(model_dir),
        local_files_only=True,
    )

    model = AutoModel.from_pretrained(
        str(model_dir),
        local_files_only=True,
    )

    model.to(_DEVICE)
    model.eval()

    _MODEL = model
    _TOKENIZER = tokenizer
    _LOADED_FROM = str(model_dir)

    return _MODEL, _TOKENIZER


def embed(texts: List[str]) -> np.ndarray:
    """Embed a list of texts into L2-normalized float32 vectors."""
    if not isinstance(texts, list):
        raise TypeError("texts must be a list of strings")

    if len(texts) == 0:
        return np.zeros((0, EMBEDDING_DIM), dtype=np.float32)

    cleaned_texts = [
        text if isinstance(text, str) and text.strip() else " "
        for text in texts
    ]

    model, tokenizer = load_bundle()
    device = _DEVICE if _DEVICE is not None else torch.device("cpu")

    encoded = tokenizer(
        cleaned_texts,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt",
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        token_embeddings = outputs.last_hidden_state
        attention_mask = encoded["attention_mask"].unsqueeze(-1).float()

        summed = (token_embeddings * attention_mask).sum(dim=1)
        counts = attention_mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts

        normalized = F.normalize(pooled, p=2, dim=1)

    return normalized.detach().cpu().numpy().astype(np.float32)


def similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    a = np.asarray(a, dtype=np.float32).reshape(-1)
    b = np.asarray(b, dtype=np.float32).reshape(-1)

    if a.shape != b.shape:
        raise ValueError(f"Shape mismatch: {a.shape} vs {b.shape}")

    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    if a_norm == 0 or b_norm == 0:
        return 0.0

    return float(np.dot(a, b) / (a_norm * b_norm))


def info() -> dict:
    """Return metadata about the frozen encoder bundle."""
    model_dir = _resolve_model_dir()
    metadata_path = model_dir.parent / "metadata.json"

    metadata = {}
    if metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

    return {
        "model_name": metadata.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"),
        "model_revision": metadata.get("model_revision"),
        "embedding_dim": EMBEDDING_DIM,
        "max_seq_len": MAX_SEQ_LEN,
        "framework": "transformers",
        "device": str(_DEVICE) if _DEVICE is not None else "not_loaded",
        "bundle_dir": str(model_dir),
    }


if __name__ == "__main__":
    import argparse
    import sys

    p = argparse.ArgumentParser(description="Bundle embed CLI")
    p.add_argument("--text", action="append", default=[], help="repeatable text input")
    p.add_argument("--texts-file", help="JSON list of strings")
    p.add_argument("--out", help="optional .npy output path")
    p.add_argument("--info", action="store_true", help="print info and exit")
    args = p.parse_args()

    if args.info:
        print(json.dumps(info(), indent=2, default=str))
        raise SystemExit(0)

    texts = list(args.text)

    if args.texts_file:
        with open(args.texts_file, encoding="utf-8") as f:
            texts.extend(json.load(f))

    if not texts:
        print("ERROR: provide --text or --texts-file", file=sys.stderr)
        raise SystemExit(2)

    emb = embed(texts)

    if args.out:
        np.save(args.out, emb)
        print(f"Saved {emb.shape} to {args.out}")
    else:
        print(json.dumps([[round(float(x), 6) for x in row] for row in emb]))
''', encoding="utf-8")

print("Wrote:", predict_path)

# Reload module cleanly after rewriting predict.py
import sys
import importlib

sys.path.insert(0, str(Path("bundle").resolve()))

if "predict" in sys.modules:
    del sys.modules["predict"]

from predict import load_bundle, embed, similarity, info, EMBEDDING_DIM, BUNDLE_DIR

model, tokenizer = load_bundle()

print("Model loaded:", type(model).__name__)
print("Tokenizer loaded:", type(tokenizer).__name__)
print("BUNDLE_DIR:", BUNDLE_DIR)
print("EMBEDDING_DIM:", EMBEDDING_DIM)
print(info())

test_embeddings = embed(["hello world", "hello world", "completely different text"])

print("Embedding shape:", test_embeddings.shape)
print("Embedding dtype:", test_embeddings.dtype)

sim_same = similarity(test_embeddings[0], test_embeddings[1])
sim_diff = similarity(test_embeddings[0], test_embeddings[2])

print("Similarity same text:", sim_same)
print("Similarity different text:", sim_diff)

assert test_embeddings.shape == (3, 384)
assert test_embeddings.dtype == np.float32
assert sim_same > sim_diff

print("Step 4 passed.")

Wrote: bundle/predict.py


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: BertModel
Tokenizer loaded: BertTokenizer
BUNDLE_DIR: /home/alireza/PycharmProjects/Quera-Homeworks/HW3_Student/HW3_A/bundle/model
EMBEDDING_DIM: 384
{'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'model_revision': '1110a243fdf4706b3f48f1d95db1a4f5529b4d41', 'embedding_dim': 384, 'max_seq_len': 256, 'framework': 'transformers', 'device': 'cuda', 'bundle_dir': '/home/alireza/PycharmProjects/Quera-Homeworks/HW3_Student/HW3_A/bundle/model'}
Embedding shape: (3, 384)
Embedding dtype: float32
Similarity same text: 1.0000001192092896
Similarity different text: 0.16112135350704193
Step 4 passed.


# Step 5: Run tests

Run all 4 test files. All tests must pass with green dots.

Tests check:
- **test_parity.py** (7 tests): Correct embedding shape, L2 normalization, similarity
- **test_tokenization.py** (5 tests): Tokenizer behavior, special tokens, round-trip
- **test_determinism.py** (1 test): Same input → same output every time
- **test_adversarial.py** (10 tests): Edge cases (unicode, numerics, long text, empty strings)

In [6]:
# Run all tests
#!cd HW3_A && PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

# Or if you're already in the HW3_A directory:
!PYTHONPATH=bundle python -m pytest tests/ -v --tb=short

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-9.0.3, pluggy-1.6.0 -- /home/alireza/PycharmProjects/Quera-Homeworks/.venv/bin/python
cachedir: .pytest_cache
rootdir: /home/alireza/PycharmProjects/Quera-Homeworks/HW3_Student/HW3_A
plugins: anyio-4.13.0, hydra-core-1.3.2
collected 23 items                                                             

tests/test_adversarial.py::test_missing_bundle_dir PASSED                [  4%]
tests/test_adversarial.py::test_very_long_text PASSED                    [  8%]
tests/test_adversarial.py::test_unicode_text PASSED                      [ 13%]
tests/test_adversarial.py::test_numeric_text PASSED                      [ 17%]
tests/test_adversarial.py::test_single_token_text PASSED                 [ 21%]
tests/test_adversarial.py::test_duplicate_texts PASSED                   [ 26%]
tests/test_adversarial.py::test_batch_of_one PASSED                      [ 30%]
tests/test_adve

# Step 6: Register in MLflow

Log the bundle to MLflow using `mlflow.sentence_transformers.log_model()`.

Before running this cell, make sure:
1. You have `source .env` (or set the env vars manually)
2. All tests pass
3. `bundle/model/` contains the 6 model files

In [14]:
# TODO: Register the bundle in MLflow

import os
import json
import hashlib
from pathlib import Path

import mlflow

# ---------------------------------------------------------------------
# 1) Load .env manually, because Jupyter may not inherit terminal env vars
# ---------------------------------------------------------------------

def load_env_file(env_path: str = ".env") -> None:
    env_file = Path(env_path)

    if not env_file.exists():
        print(f"WARNING: {env_file} not found. Using already-loaded environment variables.")
        return

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")

        os.environ[key] = value


load_env_file(".env")


# ---------------------------------------------------------------------
# 2) Validate required files and environment variables
# ---------------------------------------------------------------------

BUNDLE_DIR = Path("bundle")
BUNDLE_MODEL_DIR = BUNDLE_DIR / "model"
MANIFEST_PATH = BUNDLE_DIR / "MANIFEST.json"
METADATA_PATH = BUNDLE_DIR / "metadata.json"
PREDICT_PATH = BUNDLE_DIR / "predict.py"
REQUIREMENTS_PATH = BUNDLE_DIR / "requirements.txt"

required_bundle_files = [
    MANIFEST_PATH,
    METADATA_PATH,
    PREDICT_PATH,
    REQUIREMENTS_PATH,
    BUNDLE_MODEL_DIR / "config.json",
    BUNDLE_MODEL_DIR / "tokenizer_config.json",
    BUNDLE_MODEL_DIR / "tokenizer.json",
    BUNDLE_MODEL_DIR / "vocab.txt",
    BUNDLE_MODEL_DIR / "special_tokens_map.json",
    BUNDLE_MODEL_DIR / "model.safetensors",
]

missing_files = [str(path) for path in required_bundle_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Bundle is incomplete. Missing files:\n" + "\n".join(missing_files)
    )

required_env = [
    "MLFLOW_TRACKING_URI",
    "MLFLOW_TRACKING_USERNAME",
    "MLFLOW_TRACKING_PASSWORD",
]

missing_env = [key for key in required_env if not os.environ.get(key)]
if missing_env:
    raise EnvironmentError(
        "Missing required MLflow environment variables:\n"
        + "\n".join(missing_env)
        + "\n\nCreate/fill .env first, then rerun this cell."
    )


# ---------------------------------------------------------------------
# 3) Read metadata/manifest
# ---------------------------------------------------------------------

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

manifest_hash = hashlib.sha256(
    MANIFEST_PATH.read_bytes()
).hexdigest()

MODEL_ID = metadata.get(
    "model_name",
    os.environ.get("EMBEDDING_MODEL_ID", "sentence-transformers/all-MiniLM-L6-v2"),
)

EMBEDDING_DIM = int(metadata.get("embedding_dim", os.environ.get("EMBEDDING_DIM", 384)))
MAX_SEQ_LEN = int(metadata.get("max_seq_len", os.environ.get("EMBEDDING_MAX_SEQ_LEN", 256)))

student_username = os.environ.get("STUDENT_USERNAME", "student_unknown")
experiment_name = os.environ.get(
    "MLFLOW_EXPERIMENT_NAME",
    f"qbc12_hw03_encoder_{student_username}_2",
)
registered_model_name = os.environ.get(
    "MODEL_NAME",
    f"qbc12_hw03_encoder_{student_username}",
)


# ---------------------------------------------------------------------
# 4) Connect to MLflow and log the bundle
# ---------------------------------------------------------------------

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="hw3a-bundle") as run:
    run_id = run.info.run_id

    # Parameters
    mlflow.log_param("model_id", MODEL_ID)
    mlflow.log_param("embedding_dim", EMBEDDING_DIM)
    mlflow.log_param("max_seq_len", MAX_SEQ_LEN)
    mlflow.log_param("bundle_file_count", len(manifest["files"]))

    # Tags
    mlflow.set_tag("stage", "candidate")
    mlflow.set_tag("student_username", student_username)
    mlflow.set_tag("bundle_type", "encoder")
    mlflow.set_tag("framework", "transformers")
    mlflow.set_tag("pooling", "mean_pooling")
    mlflow.set_tag("normalization", "l2")
    mlflow.set_tag("manifest_sha256", manifest_hash)
    mlflow.set_tag("model_name", registered_model_name)

    # Log important files individually
    mlflow.log_artifact(str(METADATA_PATH), artifact_path="bundle")
    mlflow.log_artifact(str(MANIFEST_PATH), artifact_path="bundle")
    mlflow.log_artifact(str(PREDICT_PATH), artifact_path="bundle")
    mlflow.log_artifact(str(REQUIREMENTS_PATH), artifact_path="bundle")

    # Log the full bundle directory, including bundle/model/
    mlflow.log_artifacts(str(BUNDLE_DIR), artifact_path="bundle")

    artifact_uri = mlflow.get_artifact_uri("bundle")

print("MLflow registration/logging complete.")
print("Experiment name:", experiment_name)
print("Registered/model name tag:", registered_model_name)
print("Run ID:", run_id)
print("Bundle artifact URI:", artifact_uri)
print("Manifest SHA256:", manifest_hash)
print("Check MLflow UI for the run and take the screenshot for EVIDENCE/mlflow_registered.png")

2026/06/20 19:52:52 INFO mlflow.tracking.fluent: Experiment with name 'qbc12_hw03_encoder_alireza_abouei_2' does not exist. Creating a new experiment.


🏃 View run hw3a-bundle at: http://185.50.38.163:33014/#/experiments/58/runs/7d98dbfab3a84b909b673ddd124b44b4
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/58
MLflow registration/logging complete.
Experiment name: qbc12_hw03_encoder_alireza_abouei_2
Registered/model name tag: qbc12_hw03_encoder_alireza_abouei
Run ID: 7d98dbfab3a84b909b673ddd124b44b4
Bundle artifact URI: mlflow-artifacts:/58/7d98dbfab3a84b909b673ddd124b44b4/artifacts/bundle
Manifest SHA256: 2cb9dc5367e4bcb7d219c81e571d4a4ed336b1be14db972f43392e9b750024d9
Check MLflow UI for the run and take the screenshot for EVIDENCE/mlflow_registered.png


# Step 7: Upload to MinIO

Upload your `bundle/` directory to the shared MinIO bucket.

Run the provided upload script:
```bash
source .env && bash scripts/01_upload_to_minio.sh
```

Or from this notebook:

In [19]:
# Upload bundle to MinIO using the provided script
# Make sure .env is sourced first!
!bash scripts/01_upload_to_minio.sh

# Take a screenshot of the upload confirmation for EVIDENCE/minio_upload.png

Configuring MinIO alias 'qbc12'...
]11;?\Bucket: hw03-bundles
Preparing clean bundle copy...
Uploading bundle/ -> s3://hw03-bundles/alireza_abouei/
...ements.txt: 87.33 MiB / 87.33 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 408.77 KiB/s 3m38s
=== Verification ===
Files on MinIO:
[2026-06-20 21:10:49 +0330] 2.1KiB STANDARD MANIFEST.json
[2026-06-20 21:10:49 +0330]   326B STANDARD metadata.json
[2026-06-20 21:10:49 +0330]     1B STANDARD model/.cache/huggingface/.gitignore
[2026-06-20 21:10:49 +0330]   191B STANDARD model/.cache/huggingface/CACHEDIR.TAG
[2026-06-20 21:10:49 +0330]   100B STANDARD model/.cache/huggingface/download/config.json.metadata
[2026-06-20 21:10:49 +0330]   124B STANDARD model/.cache/huggingface/download/model.safetensors.metadata
[2026-06-20 21:10:49 +0330]   101B STANDARD model/.cache/huggingface/download/special_tokens_map.json.metadata
[2026-06-20 21:10:49 +0330]   101B STANDARD model/.cache/huggingface/download/tokenizer.json.metadata
[2026-06-20 21:10:49 +0330]   101B ST

## Submission Checklist

Before submitting, verify:

- [ ] `encoder_bundle.ipynb` — all cells executed with visible outputs
- [ ] `bundle/predict.py` — 4 functions implemented
- [ ] `bundle/metadata.json` — all fields filled (no TODO placeholders)
- [ ] `bundle/MANIFEST.json` — real SHA-256 hashes for every file
- [ ] `bundle/requirements.txt` — pinned dependencies listed
- [ ] `bundle/model/` — 6 model files present
- [ ] `EVIDENCE/pytest_pass.png` — all tests green
- [ ] `EVIDENCE/mlflow_registered.png` — MLflow UI showing the model
- [ ] `EVIDENCE/minio_upload.png` — MinIO upload confirmation

Good luck!